In [100]:
import os
import pandas as pd
import numpy as np
import json
import glob

In [101]:
# List all relative json paths in the current directory and its subdirectories
dataset_folder = f'mvp_aos_augment'
dataset_type = f'hoasa_hotel'
lang = 'indo'
parent_dir = f'../outputs/evals/{dataset_type}/{lang}/{dataset_folder}'
json_files = []
for root, dirs, files in os.walk(parent_dir):
	for file in files:
		if file.endswith('.json'):
			json_files.append(os.path.join(root, file))
# Sort the files by their names
json_files.sort()

In [102]:
# Discard training data evaluations
json_files = [f for f in json_files if 'unconstrained_decoding' not in f]
# json_files = [f for f in json_files if 'full_sft_traindata' not in f]
# json_files = [f for f in json_files if ('topk' in f) and ('full_sft' not in f) and ('exclude' not in f)]
# json_files = [f for f in json_files if ('topk' in f) and ('full_sft' not in f) and ('exclude_random_topk-5000' in f)]
# json_files = [f for f in json_files if ('topk' not in f) and ('full_sft' in f)]
# json_files = [f for f in json_files if (('topk-1000' in f) or ('topk-2000' in f)) and ('10-22' in f)]
# json_files = [f for f in json_files if ('full_sft' not in f)]
# json_files = [f for f in json_files if ('2025-10-29 18' in f)]
len(json_files)

15

In [103]:
# Sort json_files based on file created (newest first)
json_files = sorted(json_files, key=os.path.getctime, reverse=True)
json_files


['../outputs/evals/hoasa_hotel/indo/mvp_aos_augment/seed_31415/20251125_213920_train_model-Qwen2.5-0.5B_lr-5e-05_bs-4_epochs-10/checkpoint-24890/checkpoint-24890/constrained_decoding/inference_results.json',
 '../outputs/evals/hoasa_hotel/indo/mvp_aos_augment/seed_31415/20251125_213920_train_model-Qwen2.5-0.5B_lr-5e-05_bs-4_epochs-10/checkpoint-24890/checkpoint-24890/constrained_decoding/evaluation_results.json',
 '../outputs/evals/hoasa_hotel/indo/mvp_aos_augment/seed_31415/20251125_213920_train_model-Qwen2.5-0.5B_lr-5e-05_bs-4_epochs-10/checkpoint-24890/checkpoint-24890/constrained_decoding/raw_inference_results.json',
 '../outputs/evals/hoasa_hotel/indo/mvp_aos_augment/seed_777/20251125_213920_train_model-Qwen2.5-0.5B_lr-5e-05_bs-4_epochs-10/checkpoint-24890/checkpoint-24890/constrained_decoding/inference_results.json',
 '../outputs/evals/hoasa_hotel/indo/mvp_aos_augment/seed_777/20251125_213920_train_model-Qwen2.5-0.5B_lr-5e-05_bs-4_epochs-10/checkpoint-24890/checkpoint-24890/const

In [104]:
# # MvP
# data = {
# 	'p_aos': [],
# 	'p_aso': [],
# 	'p_sao': [],
# 	'p_oas': [],
# 	'p_osa': [],
# 	'r_aos': [],
# 	'r_aso': [],
# 	'r_sao': [],
# 	'r_oas': [],
# 	'r_osa': [],
# 	'f_aos': [],
# 	'f_aso': [],
# 	'f_sao': [],
# 	'f_oas': [],
# 	'f_osa': [],
# }
# indexes = []
# for json_file in json_files:
# 	seed = json_file.split('/')[6]
# 	topk = json_file.split('/')[8]
# 	print(seed, topk)
# 	# break
# 	if 'evaluation_result' in json_file:
# 		with open(json_file, 'r') as f:
# 			eval_data = json.load(f)
# 			data['p_aos'].append(eval_data['precision_aos'])
# 			data['p_aso'].append(eval_data['precision_aso'])
# 			data['p_sao'].append(eval_data['precision_sao'])
# 			data['p_oas'].append(eval_data['precision_oas'])
# 			data['p_osa'].append(eval_data['precision_osa'])
# 			data['r_aos'].append(eval_data['recall_aos'])
# 			data['r_aso'].append(eval_data['recall_aso'])
# 			data['r_sao'].append(eval_data['recall_sao'])
# 			data['r_oas'].append(eval_data['recall_oas'])
# 			data['r_osa'].append(eval_data['recall_osa'])
# 			data['f_aos'].append(eval_data['f1_aos'])
# 			data['f_aso'].append(eval_data['f1_aso'])
# 			data['f_sao'].append(eval_data['f1_sao'])
# 			data['f_oas'].append(eval_data['f1_oas'])
# 			data['f_osa'].append(eval_data['f1_osa'])
# 		if 'topk' in json_file:
# 			indexes.append(f"{seed}_{topk}")
# 		else:
# 			indexes.append(seed)

In [105]:
# GAS
data = {
	'p_aos': [],
	'r_aos': [],
	'f_aos': [],
}
indexes = []
for json_file in json_files:
	seed = json_file.split('/')[6]
	topk = json_file.split('/')[8]
	print(seed, topk)
	# break
	if 'evaluation_result' in json_file:
		with open(json_file, 'r') as f:
			eval_data = json.load(f)
			data['p_aos'].append(eval_data['precision_aos'])
			data['r_aos'].append(eval_data['recall_aos'])
			data['f_aos'].append(eval_data['f1_aos'])
		if 'topk' in json_file:
			indexes.append(f"{seed}_{topk}")
		else:
			indexes.append(seed)

seed_31415 checkpoint-24890
seed_31415 checkpoint-24890
seed_31415 checkpoint-24890
seed_777 checkpoint-24890
seed_777 checkpoint-24890
seed_777 checkpoint-24890
seed_123 checkpoint-24890
seed_123 checkpoint-24890
seed_123 checkpoint-24890
seed_2024 checkpoint-24890
seed_2024 checkpoint-24890
seed_2024 checkpoint-24890
seed_9584 checkpoint-24890
seed_9584 checkpoint-24890
seed_9584 checkpoint-24890


In [106]:
df = pd.DataFrame(data, index=indexes)

# Create temporary columns for sorting
df['seed'] = df.index.str.split('_').str[1]
df['topk'] = df.index.str.split('_' if 'exclude' not in json_files[0] else '-').str[-1].str.replace('topk', '').astype(int)

# Sort by seed (alphabetically) then by topk (numerically)
df = df.sort_values(['seed', 'topk'])

# Drop the temporary columns
df = df.drop(columns=['seed', 'topk'])

df

,p_aos,r_aos,f_aos
seed_123,69.309725,69.043967,69.176591
seed_2024,68.937468,69.325153,69.130767
seed_31415,68.050955,68.277096,68.163838
seed_777,68.818857,68.660532,68.739603
seed_9584,68.813646,69.095092,68.954082


In [107]:
df.to_csv('eval.csv', index=False)